# Module 2.2: Positional Encoding

In the previous notebook, we turned words into dense embedding vectors. Our model now knows the meaning of a word, but it still has a huge blindspot: **it has no concept of order or time**.

## 1. The "Bag of Words" Problem

### The Concept
If you feed a raw neural network the sentence "Dog bites man" and "Man bites dog", the mathematical operations will evaluate exactly the same. Standard attention mechanisms look at all words simultaneously in parallel.

### Why do we need to fix it?
Language is sequential. The meaning of a sentence changes entirely based on word order. Since the Transformer processes the entire sentence at once for extreme speed (unlike older RNNs which read word-by-word), we have to manually inject a "timestamp" or "position tag" directly into the word's DNA (its embedding vector).

## 2. The 2017 Solution: Absolute Sine/Cosine Encoding

### The Analogy (The Theater Tickets)
Imagine a crowd of people walking into a movie theater. To make sure they sit in the right order, you hand them a ticket. 

Why not just give them a ticket with an integer `1, 2, 3, 4`?
- Because sentences can be 10,000 words long. If you add the number `10000` to a delicate neural network embedding `[-0.1, 0.5, 0.2]`, it will completely destroy the mathematical balance of the network.

Instead, the creators of the Transformer decided the "ticket" should be a unique **chord of music**. A chord is made of continuous waves (Sine and Cosine). Every seat in the theater gets a unique visual wavelength pattern between `-1` and `1`.

### Why do we need it?
Using oscillating mathematical waves solves three massive problems:
1. The numbers never blow up (always bounded between -1 and 1).
2. It works for any sequence length (you can always calculate the sine wave further down the line).
3. The network can easily learn to calculate *relative* distances because shifting a sine wave gives you another predictable wave.

### The formula, read symbol by symbol

The "musical chord" for a position isn't hand-wavy — it comes from this pair of
equations, one filling the **even** dimensions of the vector, one the **odd**:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{\,2i/d}}\right)
\qquad
PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{\,2i/d}}\right)$$

How to read it, piece by piece:

- $pos$ — the word's **position** in the sentence (0, 1, 2, …).
- $i$ — **which pair of dimensions** we're filling (pair 0, pair 1, …). Dimension
  $2i$ gets the sine; dimension $2i+1$ gets the cosine.
- $d$ — the embedding size ($d_{model}$).
- $10000^{\,2i/d}$ — the **wavelength** of this dimension. For small $i$ it is ≈ 1
  (a fast wave — the clock's "seconds hand"); for large $i$ it is huge (a slow wave
  — the "hour hand").

So every dimension is a wave, and its speed is set by $i$. A position's full
"fingerprint" is the whole stack of these sine/cosine values. Let's compute a few
by hand before building the layer.

In [ ]:
import math

d = 128          # embedding size
pos = 1          # the word at position 1
for i in [0, 1, 32, 63]:                     # a few dimension-pairs, fast to slow
    wavelength = 10000 ** (2 * i / d)
    s = math.sin(pos / wavelength)
    c = math.cos(pos / wavelength)
    print(f"pair i={i:2d}: dim {2*i:3d} = sin = {s:+.4f} | dim {2*i+1:3d} = cos = {c:+.4f}"
          f"   (wavelength {wavelength:,.1f})")

print("\nLow pairs (i small) have wavelength ~1 -> the value swings a lot between")
print("neighbouring positions (fine detail). High pairs barely move (coarse position).")

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

# Reproducibility: identical random embeddings on every run.
torch.manual_seed(0)

# A PyTorch implementation of the classic Sine/Cosine Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 100):
        super().__init__()

        # Create a matrix of zeros [Max_Length, Dimensions]
        pe = torch.zeros(max_len, d_model)

        # Create a column of positions [0, 1, 2, 3...]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # --- The frequencies (div_term) ---
        # Mathematically we want each dimension pair i to use frequency 1 / 10000^(2i/d).
        # Computing 10000^(...) directly can overflow, so we use the identity
        #     a^x = exp(x * log(a))
        # and rewrite it as exp( (2i/d) * -log(10000) ), which is numerically stable.
        # Low dimensions  -> small i -> HIGH frequency (waves oscillate fast).
        # High dimensions -> large i -> LOW frequency  (waves oscillate slowly).
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Apply Sine to even columns, Cosine to odd columns
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe) # Save it to memory without training it

    def forward(self, x):
        # Simply ADD the positional "waves" to the word embeddings
        seq_len = x.size(1)
        return x + self.pe[:seq_len, :]

# Let's visualize what the "Theater Tickets" look like for 50 words with 128 traits.
# We feed zeros here so the heatmap shows the PURE positional pattern (no word meaning mixed in).
pe_layer = PositionalEncoding(d_model=128, max_len=50)
dummy_word_embeddings = torch.zeros(1, 50, 128) # Fake batch of zeros
positioned_words = pe_layer(dummy_word_embeddings)

plt.figure(figsize=(10, 6))
plt.pcolormesh(positioned_words[0].numpy(), cmap='viridis')
plt.xlabel('Embedding Dimensions (The 128 "Traits")')
plt.ylabel('Word Position in Sentence (0 to 49)')
plt.title('Absolute Positional Encoding Matrix (Sine/Cosine)')
plt.colorbar()
plt.show()

### How to read that heatmap

Each **row** is one word position (0 at the top, 49 at the bottom). Each **column** is one of the 128 dimensions. Look for:

- **Left columns oscillate fast.** These are the high-frequency dimensions — the color flips between yellow and purple every couple of rows. They act like the "seconds hand" of a clock, encoding fine-grained, nearby position differences.
- **Right columns change slowly.** These are low-frequency dimensions, like the "hour hand" — they barely change from one row to the next and capture coarse, long-range position.
- **Every row is a unique pattern.** No two positions share the same combination of fast and slow values, so the model can always tell positions apart — that combination *is* the position's fingerprint.

### Why is *adding* a bounded wave safe (and cheap)?

Notice we **add** the positional pattern to the embedding rather than gluing it on as extra columns (concatenation). Because each sine/cosine value lives in `[-1, 1]`, the position signal can only nudge the embedding by a small, bounded amount — it never explodes the way adding a raw integer like `10000` would. And because we add instead of concatenate, the vector keeps its original size `d_model`, so the rest of the network costs exactly the same. Free position info, no extra width.

### Position + meaning, together

The heatmap above used zeros so you could see the *pure* position pattern. In a real model the input is a meaningful word embedding, and the position wave is added on top. Let's make that visible.

In [ ]:
# Start from a REAL (here: random, standing in for "trained") word embedding,
# not zeros. Then add the position wave and watch the vector change.
real_embedding = torch.randn(1, 4, 128)   # 1 sentence, 4 words, 128-dim
with_position = pe_layer(real_embedding)

print("Same word vector, dims 0..5:")
print("  before adding position:", real_embedding[0, 0, :6])
print("  after  adding position:", with_position[0, 0, :6])

# The same word at a DIFFERENT position gets a different final vector,
# even though the underlying meaning vector is identical.
same_word = torch.randn(128)
pe = pe_layer.pe  # the [max_len, 128] table of position waves
at_pos0 = same_word + pe[0]
at_pos3 = same_word + pe[3]
print("\nIdentical word placed at position 0 vs position 3 (dims 0..5):")
print("  position 0:", at_pos0[:6])
print("  position 3:", at_pos3[:6])
print("\nThe meaning vector was the same; position alone made them differ.")

## 3. The Modern Era: Rotary Positional Embeddings (RoPE) 🚀

### The Concept
While the 2017 Sine/Cosine method (Absolute Encoding) *adds* the position to the embedding, modern models like Llama 3 and Mistral use **Rotary Positional Embeddings (RoPE)**.

### The Analogy (The Clock Hand)
Imagine your word embedding is the hand of a clock pointing at 12 o'clock.

Instead of *adding* a ticket number to the embedding, RoPE says: "Take the embedding vector and physically **rotate** it by an angle proportional to its position in the sentence."
- A word at position 1 is rotated by `1 × θ`.
- A word at position 2 is rotated by `2 × θ`.
- A word at position `m` is rotated by `m × θ`.

(Just like Sine/Cosine encoding, the vector is split into 2-D pairs and **each pair uses its own angle θ** — low dimensions rotate fast, high dimensions rotate slowly. So it is *not* a single "10 degrees" for the whole vector.)

### Why do we need it?
RoPE elegantly solves the problem of **relative distance**. When the model compares two words via a dot product, the absolute rotations cancel out and only the **difference** `(m − n)` survives. So the similarity between a word at position `m` and a word at position `n` depends only on *how far apart they are*, not on where they sit in the document. Words at positions 1 and 2 look as "2-apart" as words at positions 1001 and 1002.

That relative-only property is exactly what lets these models generalize to very long contexts (books, codebases) far beyond the lengths they trained on.

The repo already ships a production-style RoPE implementation (using complex numbers) in `src/llm_workout/layers.py` — see `precompute_freqs_cis` and `apply_rotary_emb`. Below we build the *minimal* 2-D version by hand so you can watch the relative-offset property fall out of the math.

In [ ]:
def rotate_2d(vec, position, theta):
    """Rotate a 2-D vector by (position * theta) radians."""
    angle = position * theta
    c, s = math.cos(angle), math.sin(angle)
    rot = torch.tensor([[c, -s],
                        [s,  c]])
    return rot @ vec

theta = 0.5  # the per-pair frequency (in radians)

# Two fixed 2-D "query" and "key" sub-vectors (the meaning, before any rotation).
q = torch.tensor([1.0, 0.0])
k = torch.tensor([0.7, 0.7])

print("Dot product of q (at position m) and k (at position n):\n")
print(f"{'m':>4} {'n':>4} {'m-n':>5} | dot product")
print("-" * 35)
for (m, n) in [(2, 0), (5, 3), (10, 8), (101, 99), (3, 0), (4, 0)]:
    q_rot = rotate_2d(q, m, theta)
    k_rot = rotate_2d(k, n, theta)
    dot = torch.dot(q_rot, k_rot).item()
    print(f"{m:>4} {n:>4} {m-n:>5} | {dot:+.4f}")

print("\nNotice: every row with the SAME (m-n) gives the SAME dot product,")
print("no matter how large m and n are. The absolute positions cancel out;")
print("only the RELATIVE offset survives. That is the whole point of RoPE.")

### From the 2-D toy to the real thing (complex numbers)

Real models don't rotate one 2-D vector — they rotate a whole **head vector** (e.g. 128 dimensions) by splitting it into 64 adjacent **pairs**, and giving *each pair its own frequency* $\theta_i = 10000^{-2i/d}$ (fast-spinning low pairs, slow-spinning high pairs — same frequency ladder as the Sine/Cosine table above).

And there's a beautiful shortcut for "rotate a 2-D pair": treat the pair $(a, b)$ as the **complex number** $a + bi$. Multiplying by the unit complex number $e^{i m\theta} = \cos(m\theta) + i\sin(m\theta)$ *is* rotation by $m\theta$ — one complex multiply replaces the whole rotation matrix.

So the entire production RoPE is just: *view pairs as complex numbers → multiply by precomputed $e^{im\theta_i}$ → view back as floats.* Let's prove it's identical to our `rotate_2d` loop:

In [ ]:
def precompute_freqs_cis(dim, end, theta=10000.0):
    """e^(i * m * theta_i) for every position m and every pair i -- as complex numbers."""
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))  # theta_i per pair
    t = torch.arange(end).float()                                     # positions m
    return torch.polar(torch.ones(end, dim // 2), torch.outer(t, freqs))

head_dim, seq_len = 8, 6
x = torch.randn(seq_len, head_dim)              # a tiny "query" per position

# --- THE FAST WAY (what Llama and src/llm_workout/layers.py actually do) ---
freqs_cis = precompute_freqs_cis(head_dim, seq_len)
x_complex = torch.view_as_complex(x.reshape(seq_len, head_dim // 2, 2))  # pairs -> complex
x_fast = torch.view_as_real(x_complex * freqs_cis).flatten(1)            # rotate -> floats

# --- THE SLOW WAY (your rotate_2d from above, pair by pair) ---
x_slow = torch.zeros_like(x)
for m in range(seq_len):                        # each position...
    for i in range(head_dim // 2):              # ...each 2-D pair, with its own frequency
        theta_i = 1.0 / (10000 ** (2 * i / head_dim))
        x_slow[m, 2*i : 2*i+2] = rotate_2d(x[m, 2*i : 2*i+2], m, theta_i)

print("complex fast path == rotate_2d loop?", torch.allclose(x_fast, x_slow, atol=1e-5))
print("\nThis IS the implementation inside src/llm_workout/layers.py --")
print("`precompute_freqs_cis` + `apply_rotary_emb` (names borrowed from Meta's own")
print("Llama source, so that file will feel familiar when you read it in Module 9.4).")

In [ ]:
# Task 1: a very long context still produces bounded position signals.
big_pe = PositionalEncoding(d_model=128, max_len=5000)
big_input = torch.zeros(1, 5000, 128)
big_out = big_pe(big_input)
print("Output shape:", big_out.shape)
print("Min / max value anywhere in the position table:",
      big_pe.pe.min().item(), "/", big_pe.pe.max().item())
# Your explanation here: because every value stays within [-1, 1], position 4999
# nudges the embedding by at most ~1 per dimension -- it never blows up the way
# adding the raw integer 4999 would.

# Task 2 (bonus): extend the RoPE loop with (7, 5) and (200, 198) and compare.

### 🏋️ Try it yourself

1. **Why does big `max_len` still work?** Rebuild the Sine/Cosine layer with `PositionalEncoding(d_model=128, max_len=5000)` and feed it a `torch.zeros(1, 5000, 128)` input. It works fine. Explain in a comment *why* a position of 4999 is harmless here, whereas literally adding the integer `4999` to an embedding (as in the "Theater Tickets" warning) would wreck it. (Hint: what is the range of every value in `pe`?)
2. **(Bonus)** In the RoPE cell, add the pairs `(7, 5)` and `(200, 198)` to the loop. Confirm they print the *same* dot product as the other `m-n == 2` rows.